# CUAD Contract Chunking Strategy

## Goal:

This notebook develops a chunking strategy for long CUAD contracts so they can be processed by transformer models while preserving legal context and annotated clause spans.

In [2]:
import json
import os
cwd = os.getcwd()

print("Current working directory:", cwd)

Current working directory: c:\Users\Tuand\.vscode\Accenture 1S project


In [4]:
with open("data/train_separate_questions.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [5]:
type(data)

dict

In [6]:
data.keys()

dict_keys(['version', 'data'])

In [7]:
type(data["data"])

list

In [8]:
len(data["data"])

408

In [9]:
data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
data["data"][0]

{'title': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT',
 'paragraphs': [{'qas': [{'answers': [{'text': 'DISTRIBUTOR AGREEMENT',
       'answer_start': 44}],
     'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name_0',
     'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract',
     'is_impossible': False},
    {'answers': [{'text': 'Distributor', 'answer_start': 244}],
     'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties_0',
     'question': 'Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract',
     'is_impossible': False},
    {'answers': [{'text': 'Electric City of Illinois L.L.C.',
       'answer_start': 49574}],
     'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties_1',
     'question': 'Highlight the pa

In [11]:
first_contract = data["data"][0]

print(first_contract.keys())

dict_keys(['title', 'paragraphs'])


In [12]:
print(first_contract.get("title"))

LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT


In [13]:
type(first_contract["paragraphs"])

list

In [14]:
len(first_contract["paragraphs"])

1

In [15]:
first_contract["paragraphs"][0].keys()

dict_keys(['qas', 'context'])

## Inspect CUAD Contract Structure:

Each CUAD contract contains contract text in `context` and question-answer annotations in `qas`.
We inspect these fields before designing the chunking strategy.

In [16]:
paragraph = first_contract["paragraphs"][0]

print(paragraph["context"][:1500])

EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.

         B. Representations.  As an inducement to the Company to enter into this Agreement,  the  Distributor  has  represented  that  it has or  will  have  the facil

In [17]:
len(paragraph["context"])

54290

In [18]:
len(paragraph["qas"])

68

In [19]:
first_qa = paragraph["qas"][0]

print("Question:")
print(first_qa["question"])

print("\nAnswers:")
print(first_qa["answers"])

print("\nIs impossible:")
print(first_qa["is_impossible"])

Question:
Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract

Answers:
[{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}]

Is impossible:
False


## Analyze Contract Lengths:

Before choosing a chunk size, we examine how long the CUAD contracts are.
The `context` length above is measured in characters, but transformer models
operate on tokens, so we will later measure token counts using the model tokenizer.

In [20]:
paragraph_counts = [
    len(contract["paragraphs"])
    for contract in data["data"]
]

print("Minimum paragraphs:", min(paragraph_counts))
print("Maximum paragraphs:", max(paragraph_counts))
print("Average paragraphs:", sum(paragraph_counts) / len(paragraph_counts))

Minimum paragraphs: 1
Maximum paragraphs: 1
Average paragraphs: 1.0


In [21]:
context = paragraph["context"]

word_count = len(context.split())

print("Characters:", len(context))
print("Approximate words:", word_count)

Characters: 54290
Approximate words: 5998


In [22]:
contract_word_counts = []

for contract in data["data"]:
    full_text = " ".join(
        p["context"] for p in contract["paragraphs"]
    )
    
    contract_word_counts.append(len(full_text.split()))

print("Shortest contract:", min(contract_word_counts), "words")
print("Longest contract:", max(contract_word_counts), "words")
print("Average contract:", round(sum(contract_word_counts) / len(contract_word_counts)), "words")

Shortest contract: 170 words
Longest contract: 45650 words
Average contract: 8045 words


## Measure Contract Length in Tokens:

Word counts provide an approximate measure of contract length, but transformer
models process tokens rather than words. We therefore use a tokenizer to measure
the actual number of tokens in each contract before selecting a chunk size.

## Tokenizer Choice

We as a team have not selected a final transformer model yet.

For initial chunking experiments, I'll use the `bert-base-uncased` tokenizer as a baseline, because BERT supports inputs up to 512 tokens, so the final chunk size must leave room for special tokens and, if needed, the question/category text.

P.S. The tokenizer can be replaced later when the final model is selected.

In [23]:
%pip install transformers

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.6 MB 206.9 kB/s eta 0:00:59
   -- ------------------------------------- 0.8/12.6 MB 344.4 kB/s eta 0:00:35
   --- ------------------------------------ 1.0/12.6 MB 465.8 kB/s eta 0


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [25]:
context = paragraph["context"]

tokens = tokenizer.encode(
    context,
    add_special_tokens=False
)

print("Words:", len(context.split()))
print("Tokens:", len(tokens))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7556 > 512). Running this sequence through the model will result in indexing errors


Words: 5998
Tokens: 7556


In [26]:
contract_token_counts = []

for contract in data["data"]:
    full_text = " ".join(
        p["context"] for p in contract["paragraphs"]
    )

    token_count = len(
        tokenizer.encode(
            full_text,
            add_special_tokens=False
        )
    )

    contract_token_counts.append(token_count)

print("Shortest contract:", min(contract_token_counts), "tokens")
print("Longest contract:", max(contract_token_counts), "tokens")
print(
    "Average contract:",
    round(sum(contract_token_counts) / len(contract_token_counts)),
    "tokens"
)

Shortest contract: 224 tokens
Longest contract: 78591 tokens
Average contract: 11147 tokens


## Boundary-Aware Chunking Strategy

Chunks are capped at 450 content tokens with a 75-token overlap. The chunker
prefers sentence-ending or newline boundaries near the token limit, then falls
back to the hard limit. It returns original text slices and character/token
offsets so the same chunks can be used for training and span-coverage checks.

If category or question text is added to the model input later, reduce the
content-token cap to leave room for that text and special tokens.

In [ ]:
MAX_TOKENS = 450
OVERLAP = 75

In [42]:
import re
from bisect import bisect_right


def chunk_text(text, tokenizer, max_tokens=450, overlap=75):
    if max_tokens <= 0:
        raise ValueError("max_tokens must be positive")
    if overlap < 0 or overlap >= max_tokens:
        raise ValueError("overlap must be between 0 and max_tokens - 1")

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )
    token_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    if not token_ids:
        return []

    token_ends = [end for _, end in offsets]
    boundary_chars = [
        match.end()
        for match in re.finditer(r"\n+|(?<=[.!?])\s+", text)
    ]
    preferred_ends = sorted({
        bisect_right(token_ends, char_position)
        for char_position in boundary_chars
    })

    chunks = []
    start_token = 0
    minimum_chunk_size = max(1, max_tokens // 2)

    while start_token < len(token_ids):
        end_token = min(start_token + max_tokens, len(token_ids))

        if end_token < len(token_ids):
            boundary_index = bisect_right(preferred_ends, end_token) - 1
            while boundary_index >= 0:
                candidate_end = preferred_ends[boundary_index]
                if candidate_end < start_token + minimum_chunk_size:
                    break
                if candidate_end > start_token:
                    end_token = candidate_end
                    break
                boundary_index -= 1

        char_start = offsets[start_token][0]
        char_end = offsets[end_token - 1][1]
        chunks.append({
            "text": text[char_start:char_end],
            "start_token": start_token,
            "end_token": end_token,
            "char_start": char_start,
            "char_end": char_end,
            "num_tokens": end_token - start_token
        })

        if end_token >= len(token_ids):
            break
        start_token = max(start_token + 1, end_token - overlap)

    return chunks

In [ ]:
chunks = chunk_text(
    paragraph["context"],
    tokenizer,
    max_tokens=MAX_TOKENS,
    overlap=OVERLAP
)

print("Original contract tokens:", len(tokens))
print("Number of chunks:", len(chunks))

Original contract tokens: 7556
Number of chunks: 19


In [ ]:
print(chunks[0]["text"][:1500])

exhibit 10. 6 distributor agreement this distributor agreement ( the " agreement " ) is made by and between electric city corp., a delaware corporation ( " company " ) and electric city of illinois llc ( " distributor " ) this 7th day of september, 1999. recitals a. the company ' s business. the company is presently engaged in the business of selling an energy efficiency device, which is referred to as an " energy saver " which may be improved or otherwise changed from its present composition ( the " products " ). the company may engage in the business of selling other products or other devices other than the products, which will be considered products if distributor exercises its options pursuant to section 7 hereof. b. representations. as an inducement to the company to enter into this agreement, the distributor has represented that it has or will have the facilities, personnel, and financial capability to promote the sale and use of products. as an inducement to distributor to enter

In [ ]:
print("First chunk token count:", chunks[0]["num_tokens"])
print("First chunk character offsets:", chunks[0]["char_start"], chunks[0]["char_end"])

First chunk token count: 450


## Evaluation Experiments — 09/27/2026


## Evaluating the Chunking Strategy

A good chunking strategy should stay within the model's token limit and preserve
CUAD's annotated answer spans. This experiment evaluates candidate settings
across the official training split, keeping all chunks from each contract in
that split.

For each setting, it reports:

- complete answer-span coverage
- total and average chunks per contract
- maximum observed chunk length
- token overhead caused by overlap

In [49]:
test_chunks = chunk_text(
    paragraph["context"],
    tokenizer,
    max_tokens=MAX_TOKENS,
    overlap=OVERLAP
)

print("Number of chunks:", len(test_chunks))
print(test_chunks[0])

Number of chunks: 16
{'text': 'ENDORSEMENT AGREEMENT    entered into by and between     ANDY NORTH     and     GOLFERS INCORPORATED\n\n   Effective February 21, 2011\n\n\n\nSource: PERFORMANCE SPORTS BRANDS, INC., S-1, 9/9/2011\n\n\n\n\n\n   TABLE OF CONTENTS\n\n\n\n\n\nParagraph   Page            1.   Definitions   3  2.   Term   4  3.   Grant of License and Exclusivity   4  4.   Retention of Rights   4  5.   Appearances   4  6.   Compensation   5  7.   Supply of Endorsed Products   6  8.   Approval of Advertising   7  9.   Ownership   8  10.   SAG and/or AFTRA   8  11.   Standards   8  12.   Events of Default   8  13.   Termination/Remedies   9  14.   Company\'s Debts   9  15.   Indemnification   9  16.   Insurance   9  17.   Waiver   9  18.   Notices   10  19.   Assignment   10  20.   Independent Contractor   10  21.   Joint Venture   10  22.   Governing Law   10  23   Entire Agreement   10  24.   Amendments   10  25.   Authority   11  26.   Severability   11  27.   Compliance with 

In [50]:
print("Number of chunks:", len(test_chunks))
print("First chunk text:", test_chunks[0]["text"][:500])
print(
    "Character offsets:",
    test_chunks[0]["char_start"],
    test_chunks[0]["char_end"]
)

Number of chunks: 16
First chunk text: ENDORSEMENT AGREEMENT    entered into by and between     ANDY NORTH     and     GOLFERS INCORPORATED

   Effective February 21, 2011



Source: PERFORMANCE SPORTS BRANDS, INC., S-1, 9/9/2011





   TABLE OF CONTENTS





Paragraph   Page            1.   Definitions   3  2.   Term   4  3.   Grant of License and Exclusivity   4  4.   Retention of Rights   4  5.   Appearances   4  6.   Compensation   5  7.   Supply of Endorsed Products   6  8.   Approval of Advertising   7  9.   Ownership   8  10.
Character offsets: 0 1598


In [43]:
def get_answer_spans(contract):
    answer_spans = []

    for paragraph_index, paragraph in enumerate(contract["paragraphs"]):
        for qa in paragraph["qas"]:
            for answer in qa["answers"]:
                start = answer["answer_start"]
                end = start + len(answer["text"])

                answer_spans.append({
                    "paragraph_index": paragraph_index,
                    "question": qa["question"],
                    "text": answer["text"],
                    "start": start,
                    "end": end
                })

    return answer_spans

In [35]:
answers = get_answer_spans(first_contract)

print("Annotated answers:", len(answers))
print(answers[0])

Annotated answers: 46
{'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'text': 'DISTRIBUTOR AGREEMENT', 'start': 44, 'end': 65}


In [45]:
def answer_is_preserved(answer, chunks):
    for chunk in chunks:
        if (
            answer["start"] >= chunk["char_start"]
            and answer["end"] <= chunk["char_end"]
        ):
            return True

    return False

In [37]:
preserved = 0

for answer in answers:
    if answer_is_preserved(answer, test_chunks):
        preserved += 1

coverage = preserved / len(answers) * 100

print("Annotated answers:", len(answers))
print("Preserved answers:", preserved)
print("Coverage:", round(coverage, 2), "%")

Annotated answers: 46
Preserved answers: 42
Coverage: 91.3 %


### Initial Result

Using 450-token chunks with a 50-token overlap:

- 46 annotated answer spans were present in the first contract.
- 42 spans were fully preserved within at least one chunk.
- Answer-span coverage was **91.3%**.
- 4 annotated spans crossed chunk boundaries and were not fully preserved.

This suggests that the baseline token-based strategy works reasonably well,
but additional overlap or smarter chunk boundaries may improve preservation
of legal clauses. Next, we inspect the failed spans and compare different overlap settings.

In [38]:
failed_answers = []

for answer in answers:
    if not answer_is_preserved(answer, test_chunks):
        failed_answers.append(answer)

print("Failed answers:", len(failed_answers))

for i, answer in enumerate(failed_answers, 1):
    print(f"\nFailure {i}")
    print("Question:", answer["question"])
    print("Answer:", answer["text"])
    print("Start:", answer["start"])
    print("End:", answer["end"])

Failed answers: 4

Failure 1
Question: Highlight the parts (if any) of this contract related to "Price Restrictions" that should be reviewed by a lawyer. Details: Is there a restriction on the  ability of a party to raise or reduce prices of technology, goods, or  services provided?
Answer: The prices set forth in                            Section   2.4(a)  shall  be  subject  to   adjustment                            annually  on the  first  day  of  each  Product  Year                            beginning in the calendar  year 2000 and on the first                            day of each succeeding Product Year for the remainder                            of the Term and all  renewals  of this  Agreement  in                            proportion   to  the  increase  or  decrease  in  the                            Consumer  Price Index (CPI) as compared to the CPI as                            it  existed  on the  first  day of the  Term  of this                            Agreement

In [46]:
import pandas as pd

In [47]:
settings = [
    (400, 50),
    (450, 50),
    (450, 75),
    (450, 100)
]

results = []

for max_tokens, overlap in settings:
    total_answers = 0
    preserved_answers = 0
    total_chunks = 0
    source_tokens = 0
    chunk_tokens = 0
    max_observed_chunk_tokens = 0

    for contract in data["data"]:
        answer_spans = get_answer_spans(contract)
        answers_by_paragraph = {}
        for answer in answer_spans:
            answers_by_paragraph.setdefault(answer["paragraph_index"], []).append(answer)

        for paragraph_index, paragraph in enumerate(contract["paragraphs"]):
            text = paragraph["context"]
            paragraph_chunks = chunk_text(
                text,
                tokenizer,
                max_tokens=max_tokens,
                overlap=overlap
            )
            paragraph_answers = answers_by_paragraph.get(paragraph_index, [])

            total_answers += len(paragraph_answers)
            preserved_answers += sum(
                answer_is_preserved(answer, paragraph_chunks)
                for answer in paragraph_answers
            )
            total_chunks += len(paragraph_chunks)
            source_tokens += len(tokenizer.encode(text, add_special_tokens=False))
            chunk_tokens += sum(chunk["num_tokens"] for chunk in paragraph_chunks)
            max_observed_chunk_tokens = max(
                max_observed_chunk_tokens,
                max((chunk["num_tokens"] for chunk in paragraph_chunks), default=0)
            )

    results.append({
        "chunk_size": max_tokens,
        "overlap": overlap,
        "preserved_answers": preserved_answers,
        "total_answers": total_answers,
        "coverage_pct": round(100 * preserved_answers / total_answers, 2),
        "chunks": total_chunks,
        "avg_chunks_per_contract": round(total_chunks / len(data["data"]), 2),
        "max_observed_tokens": max_observed_chunk_tokens,
        "overlap_overhead_pct": round(
            100 * (chunk_tokens - source_tokens) / source_tokens,
            2
        )
    })

pd.DataFrame(results)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7556 > 512). Running this sequence through the model will result in indexing errors


,chunk_size,overlap,preserved_answers,total_answers,coverage_pct,chunks,avg_chunks_per_contract,max_observed_tokens,overlap_overhead_pct
0,400,50,10988,11180,98.28,14518,35.58,400,15.51
1,450,50,11036,11180,98.71,12620,30.93,450,13.43
2,450,75,11055,11180,98.88,13491,33.07,450,21.58
3,450,100,11072,11180,99.03,14517,35.58,450,31.02


### Training Split Comparison

The earlier 75-token overlap result was measured on only the first contract, so
it is a candidate rather than a final setting. Use the table above to compare
coverage and chunk overhead across all official training contracts. Keep the
official test split untouched until the chunking settings are selected.

## Fixed-Window vs Boundary-Aware Example

The next cell runs both strategies on the same contract text. The fixed-window
version reproduces the original 450-token / 50-token-overlap behavior; the new
version uses the boundary-aware chunker and its 75-token overlap.

In [51]:
text = first_contract["paragraphs"][0]["context"]

old_token_ids = tokenizer.encode(text, add_special_tokens=False)
old_overlap = 50
old_step = MAX_TOKENS - old_overlap
old_chunks = []

for start_token in range(0, len(old_token_ids), old_step):
    end_token = min(start_token + MAX_TOKENS, len(old_token_ids))
    old_chunks.append(
        tokenizer.decode(old_token_ids[start_token:end_token], skip_special_tokens=True)
    )
    if end_token >= len(old_token_ids):
        break

new_chunks = chunk_text(
    text,
    tokenizer,
    max_tokens=MAX_TOKENS,
    overlap=OVERLAP
)

comparison = pd.DataFrame([
    {
        "strategy": "Old fixed window (450 / 50)",
        "number_of_chunks": len(old_chunks),
        "first_chunk_tokens": min(MAX_TOKENS, len(old_token_ids)),
        "return_type": type(old_chunks[0]).__name__,
        "first_chunk_offsets": "not retained"
    },
    {
        "strategy": "New boundary-aware (450 / 75)",
        "number_of_chunks": len(new_chunks),
        "first_chunk_tokens": new_chunks[0]["num_tokens"],
        "return_type": type(new_chunks[0]).__name__,
        "first_chunk_offsets": (
            f"{new_chunks[0]['char_start']}:{new_chunks[0]['char_end']}"
        )
    }
])

display(comparison)

print("Old first chunk:")
print(old_chunks[0][:400])
print("\nNew first chunk:")
print(new_chunks[0]["text"][:400])

,strategy,number_of_chunks,first_chunk_tokens,return_type,first_chunk_offsets
0,Old fixed window (450 / 50),19,450,str,not retained
1,New boundary-aware (450 / 75),20,433,dict,0:2867


Old first chunk:
exhibit 10. 6 distributor agreement this distributor agreement ( the " agreement " ) is made by and between electric city corp., a delaware corporation ( " company " ) and electric city of illinois llc ( " distributor " ) this 7th day of september, 1999. recitals a. the company ' s business. the company is presently engaged in the business of selling an energy efficiency device, which is referred 

New first chunk:
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  present
